# <u>Isometric mapping and Locally linear embedding</u> 


* [0. Big Picture: Dimensionality Reduction](#bigpic)
* [1. Manifold Learning Idea](#mani)
* [2. Neighborhood graph](#neighbor)
* [3. ISOMAP aka Isometric Mapping](#isomap)
* [4. LLE aka Locally Linear Embedding](#lle)
* [5. ISOMAP library](#lib1)
* [6. LLE library](#lib2)

In [ ]:
import numpy as np # for math and random numbers
import math # for math
import pandas as pd # for dataframes
import plotly.express as px # for plotting
import plotly.graph_objects as po # for plotting
from sklearn.datasets import make_s_curve # make non linear data
from sklearn.preprocessing import StandardScaler # for standardizing
from sklearn.manifold import Isomap # for Isometric mapping
from sklearn.manifold import LocallyLinearEmbedding # for locally linear embedding
from sklearn.manifold import TSNE # from sklearn.manifold import TSNE
#import umap.umap_ as umap # Uniform Manifold Approximation and Projection
print("Setup complete")

Setup complete


In [ ]:
# uniform sampled surface of a sphere
x = np.random.randn(3, 200)
normsx = np.sqrt((x*x).sum(axis=0)) # norms of all x
x = x / normsx  
px.scatter_3d(x=x[0], y=x[1], z=x[2]) # cool

<a class="anchor" id="bigpic"></a>
## 0. Big Picture: Dimensionality Reduction

Given $n$ data points $$X = [x_1,...,x_n] \in \mathbb{R}^{D \times n} $$ find low dimensional representation $$Z=[z_1,...,z_n] \in \mathbb{R}^{d \times n} $$

with $d \ll D$ that keeps most of the structure of the higher dimensional data.

What structure should be preserved?

| Method | Preserves                        |
|--------|----------------------------------|
| PCA    | Global variance (linear)         |
| ISOMAP | Geodesic distances on a manifold |
| LLE    | Local neighborhood geometry      |

So ISOMAP and LLE keep neighborhood relations and instead of assuming data lives in a flat space, we assume it lives on a manifold.


<div style="display:flex; gap:20px;">

<div>

### Linear (z. B. PCA)
- Data lies on linear a linear structure
- Examples: Line, Plane, Hyperplane


</div>


<div>

### Nonlinear (Manifold Learning)
- Data lies on curved manifold
- Localy structure looks like $\mathbb{R}^d$, but not global
- Assumption of ISOMAP and LLE

</div>


<a class="anchor" id="mani"></a>
## 1. Manifold Learning Data

A manifold is a curved surface embedded in higher dimensions that locally looks flat.

So:

- globally nonlinear
- locally linear

ISOMAP and LLE exploit this:

- ISOMAP: preserve global geodesic distances
- LLE: preserve local linear reconstructions

<p align="center">
<img src="pics/manifold.png" width="400"/>
</p>

In [116]:
# generate the swiss roll dataset
def swissroll(n=1000, evenly_sampled=True):
    z = np.random.rand(2, n)
    if evenly_sampled:
        z[0] = np.sqrt(z[0])
    z = 3 * math.pi * z
    r = z[0] + 1.0
    x = np.vstack([r*np.cos(z[0]), 
                   r*np.sin(z[0]), 
                   z[1]])
    return x, z

x, z = swissroll()
px.scatter_3d(x=x[0], y=x[1], z=x[2], color=z[0], color_continuous_scale=px.colors.sequential.Rainbow)

<a class="anchor" id="neighbor"></a>
## 2. Neighborhood graph

Both methods ISOMAP and LLE capture the manifold by defining a Neighborhood graph:

Given $n$ data points $$X = [x_1,...,x_n] \in \mathbb{R}^{D \times n} $$ 

- nodes, vertices $x_1,...,x_n$
- edges between neighbors, i.e. close-by ponits

Examples

<div style="display:flex; gap:20px;">

<div>

### $k$ nearest neighbor graph (knn-graph)
- neighbors of $x_i$ are $k$-nearest data points of $x_i$
- $k$-nn does not depend on scaling
- Each point connects to its k closest points
- Case $k=0$
    - Each point has no neighbors
    - Graph has n isolated nodes / vertices
    - No edges
    - Graph is completely disconnected
    - Consequences:
        - ISOMAP: shortest paths are $\infty \rightarrow$  fails
        - LLE: cannot reconstruct points $\rightarrow$ fails
- Case: $k=1$
    - Each point connects only to its single nearest neighbor
    - Graph usually not fully connected
    - Geodesic distances become unreliable
    - Consequences:
        - ISOMAP produces broken embeddings
        - LLE becomes unstable
        - You don't have enough local structure to approximate a manifold
- Case: $k=n$
    - Each point connects to all other points
    - Fully connected complete graph
    - Every edge exists
    - Consequences:
        - Graph distances = Euclidean distances
        - ISOMAP reduces to classical MDS / PCA
        - Manifold assumption disappears
        - LLE becomes meaningless (global linear model)
        - No locality $\rightarrow$ no nonlinear behavior

$\Rightarrow$ Choose $k$ big enough to connect the graph, but small enough to preserve locality

Typically $5 \leq k \leq 15$ 


</div>


<div>

### $\varepsilon$ graph
- neighbors of $x_i$ are data points closer than $\varepsilon$ to $x_i$
- $\varepsilon$-graph is sensitive to the metric of the data points
- Sensitive to scaling
- Each point connects to all points within distance $\varepsilon$
- $\lVert x_i - x_j \rVert \leq \varepsilon$
- Case: $\varepsilon = 0$ 
    - Only points at exactly the same location connect
    - No edges
    - Graph disconnected
    - Consequences:
        - ISOMAP fails
        - LLE fails
- Case: Very small $\varepsilon$
    - Only extremely close points connect
    - Graph fragmented
    - Consequences:
        - Distances become $\infty$
        - Embedding breaks

- Case: Medium $\varepsilon$
    - Connected graph
    - Local
    - Consequences:
        - Manifold approximated correctly
        - ISOMAP + LLE work well

- Case: $\varepsilon \rightarrow \infty$
    - Every point connects to every other point
    - Consequences:
        - ISOMAP becomes linear method

</div>

</div>

| Feature           | kNN                 | ε-graph                |
| ----------------- | ------------------- | ---------------------- |
| Parameter         | number of neighbors | radius                 |
| Scale sensitive   | ❌ no                | &#9989; yes               |
| Connectivity      | controlled          | unpredictable          |
| Density variation | good                | bad in varying density |
| Stability         | high                | lower                  |

&#128073; kNN is usually preferred because ε depends heavily on scaling

In [117]:
def plot_graph(x, W, color=None):
    n = W.shape[0]
    edges_x, edges_y = [], []
    for i in range(n):
        for j in range(i):
            if 0 < W[i,j] < np.inf:
                edges_x.extend([x[0,i], x[0,j], None])
                edges_y.extend([x[1,i], x[1,j], None])
    edges_trace = po.Scatter(x=edges_x, y=edges_y, mode='lines')
    nodes_trace = po.Scatter(x=x[0],    y=x[1],    mode='markers', 
                             marker=dict(color=color, size=5.0))
    return po.Figure(data=[edges_trace, nodes_trace])


def plot_graph_3d(x, W, color=None):
    n = W.shape[0]
    edges_x, edges_y, edges_z = [], [], []
    for i in range(n):
        for j in range(i):
            if 0 < W[i,j] < np.inf:
                edges_x.extend([x[0,i], x[0,j], None])
                edges_y.extend([x[1,i], x[1,j], None])
                edges_z.extend([x[2,i], x[2,j], None])
    edges_trace = po.Scatter3d(x=edges_x, y=edges_y, z=edges_z, mode='lines')
    nodes_trace = po.Scatter3d(x=x[0],    y=x[1],    z=x[2],    mode='markers', 
                               marker=dict(color=color, size=5.0))
    return po.Figure(data=[edges_trace, nodes_trace])

<a class="anchor" id="isomap"></a>
## 3. ISOMAP aka Isometric Mapping

#### Core Idea
**Euclidean distance fails on curved manifolds, so instead**

0. Given data matrix $X=[x_1,...,x_n] \in \mathbb{R}^{D \times n}$ with data points as columns
1. Build neighborhood graph
2. Compute all-pairs-shortest paths along graph $\rightarrow$ geodesic distance (i.e. distance along the manifold)
3. Apply MDS to find low dimensional embedding

So ISOMAP = Graph $\rightarrow$ Geodesic distances $\rightarrow$ MDS

### ISOMAP stepwise

##### <u>Step 0: Data matrix</u>

Collect $n$ data points of shape $\mathbb{R}^D$ into columns of $X$

##### <u>Step 1: Neighborhood Graph</u>

Given $n$ data points $X = [x_1,...,x_n] \in \mathbb{R}^{D \times n}$
1. Calculate $D_{ij}=\lVert x_i-x_j \rVert$
2. For each point $x_i$ make a list of iths neighbors (e.g. from $k$-nn graph or $\varepsilon$-graph)
3. Define weights matrix with entries 
$$
W_{ij} = 
\begin{cases} 
0 \hspace{0.4cm} \text{if } i=j \\
D_{ij} \hspace{0.2cm} \text{if } x_j \text{ and } x_j \text{ are neighbors} \\
\infty \hspace{0.3cm} \text{otherwise}  
\end{cases}
$$

For $0 < W_{ij} < \infty$ there is an edge from $x_i$ to $x_j$ and 
$
\mathbb{R}^{n \times n} \ni W = 
\begin{cases} 
\text{ Symmetric for } \varepsilon\text{-graphs} \\
\text{ Not symmetric for } k\text{-nn graphs} \\ 
\end{cases}
$

##### <u>Step 2: All-pairs shortest paths</u>

Given a graph with edge weights $W_{ij}$ (their lengths) calculate a matrix $D$ of all lengths of all paths along the graph. Initialize $D=W$.

So compute $D_{ij} =$ shortest path between $i,j$

- update all lengths in $D$ with respect to paths via $x_1$
- update all lengths in $D$ with respect to paths via $x_2$
- $\ldots$
- update all lengths in $D$ with respect to paths via $x_n$

Using: 
- Floyd–Warshall
<div style="display:flex; gap:20px;">
<div>

```python
def floyd_warshall(W):    # Python/numpy: 3 python loops
    # assumes non-squared distances in W
    D = W.copy()
    n = W.shape[0]
    for k in range(n):
        for i in range(n):
            for j in range(n):
                D[i,j] = min(D[i,j], D[i,k]+D[k,j])
    return D

```
</div>
<div>

```python
def floyd_warshall(W):    # Python/numpy: 2 python loops
    # assumes non-squared distances in W
    D = W.copy()
    n = W.shape[0]
    for k in range(n):
        for i in range(n):
            D[i,:] = np.minimum(D[i,:], D[i,k]+D[k,:])
    return D

```
</div>
</div>

- Dijkstra
- or scipy graph routines

This approximates geodesic distance on the manifold.


##### <u>Step 3: MDS (Multidimensional Scaling)</u>

Apply MDS to transform distances $D$ into embedding $Z=[z_1,...,z_n]$

Given 
- data matrix $X=[x_1,...,x_n] \in \mathbb{R}^{d \times n}$ where we assume $\sum_{i=1}^n x_i = 0$ 
- and matrix $D \in \mathbb{R}^{n \times n}$ of squared distances with  
$D_{ij}=\lVert x_i - x_j \rVert^2 = (x_i - x_j)^\top (x_i - x_j) = x_i^\top x_i + x_j^\top x_j - 2x_i^\top x_j$ 

**Algortihm**

1. Calculate Gram matrix (inner product matrix) $\color{red}G=X^\top X\color{white} \in \mathbb{R}^{n \times n} \rightarrow$ Difficult
2. Calculate EVD of $G=V\Lambda V^\top$
3. Pick $d$ largest eigenvalues from $\Lambda$ and collect corresponding eigenvectors as columns into the matrix $V_d$
4. Calculate data matrix $X=\Lambda^{\frac{1}{2}} V_d^\top$

Note: 
- $X^\top X = (\Lambda^{\frac{1}{2}} V^\top)^\top \Lambda^{\frac{1}{2}} V^\top = V \Lambda^{\frac{1}{2}} \Lambda^{\frac{1}{2}} V^\top = V\Lambda V^\top = G$
- $\color{red}\mathrm{Ques}: \text{How do we calculate the Gram matrix from the distance matrix?} \rightarrow \text{ Use lemma}$

**Lemma:**
- Squared distances can be calculated from Gram matrix $G$
$$
D = (G \odot I_n) 1_n 1_n^\top + 1_n 1_n^\top (G \odot I_n) - 2G
$$
- $G \odot I_n = \text{diag}(\text{diag}(G))$ is matrix with squared norms $x_1^\top x_1, ..., x_n^\top x_n$ along diagonal
- Centering matrix $H=I_n - \frac{1}{n} 1_n 1_n^\top$ where $1_n 1_n^\top$ is ones matrix
- Assuming mean of $X$ is 0, i.e. $X1_n = 0$ then $XH = X(I_n - \frac{1}{n} 1_n 1_n^\top) = X - X\frac{1}{n} 1_n 1_n^\top = X - 0_n \frac{1}{n} 1_n^\top = X$
- Gram matrix $G = X^\top X = (XH)^\top XH = H^\top X^\top XH = HX^\top XH = HGH$

$$\Rightarrow HDH = H[(G \odot I_n) 1_n 1_n^\top + 1_n 1_n^\top (G \odot I_n) - 2G]H   \\ 
\iff HDH =  H(G \odot I_n) 1_n 1_n^\top H + H1_n 1_n^\top (G \odot I_n)H - 2HGH  \\
 1_n 1_n^\top H = 1_n 1_n^\top(I_n - \frac{1}{n} 1_n 1_n^\top) = 1_n 1_n^\top - 1_n 1_n^\top \frac{1}{n} 1_n 1_n^\top = 1_n 1_n^\top - 1_n (1_n^\top 1_n) \frac{1}{n}  1_n^\top = 1_n 1_n^\top - 1_n n \frac{1}{n}  1_n^\top = 0_{n \times n} = H1_n 1_n^\top   \\
 \iff HDH =  H(G \odot I_n)0_{n \times n} + 0_{n \times n}(G \odot I_n)H - 2HGH  \\
 \text{Assume }X\text{ has zero mean, then } XH = X \text{ and } HGH = G\\
  \iff HDH =  H(G \odot I_n)0_{n \times n} + 0_{n \times n}(G \odot I_n)H - 2G  \\
  \iff HDH = -2G  \\
  \iff\color{red} -\frac{1}{2}HDH = G
$$ 

Given 
- data matrix $X=[x_1,...,x_n] \in \mathbb{R}^{d \times n}$ where we assume $\sum_{i=1}^n x_i = 0$ 
- and matrix $D \in \mathbb{R}^{n \times n}$ of squared distances with  
$D_{ij}=\lVert x_i - x_j \rVert^2 = (x_i - x_j)^\top (x_i - x_j) = x_i^\top x_i + x_j^\top x_j - 2x_i^\top x_j$ 

**Algortihm**

1. Calculate Gram matrix (inner product matrix) $\color{red}G=-\frac{1}{2}HDH$
2. Calculate EVD of $G=V\Lambda V^\top$ and $\Lambda = \begin{bmatrix} \lambda_1 & 0 & ... & 0 \\ 0 & \lambda_2 & ... & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & ... & \lambda_D \end{bmatrix}$
3. Pick $d$ largest eigenvalues from $\Lambda$ and collect corresponding eigenvectors as columns into the matrix $V_d$
4. Calculate data matrix $X=\Lambda^{\frac{1}{2}} V_d^\top$


Given 
$$
X = [x_1,...,x_n] = \begin{bmatrix}  x_{11} & x_{12} & \ldots & x_{1n} \\ x_{21} & x_{22} & \ldots & x_{2n} \\ \vdots & \vdots & \ddots & \vdots \\ x_{D1} & x_{D2} & \ldots & x_{Dn}\end{bmatrix}   \ \in \mathbb{R}^{D \times n} 
$$

compute 
$$
D_{ij}=\lVert x_i - x_j \rVert = \sqrt{\sum_{i,j}^n(x_i - x_j)^2} \text{ for }i,y \in \{1,...,n\}
$$


In [118]:
def distances(x, y=None, rowvectors=False): # non-squared distances
    # if rowvectors==False:
    #   x is (d, m)
    #   y is (d, n)
    # otherwise the data matrices are
    #   x is (m, d)
    #   y is (n, d)
    if rowvectors:
        x = x.T
        y = y.T
    if y is None:
        y = x
    xx = (x*x).sum(axis=0, keepdims=True)
    yy = (y*y).sum(axis=0, keepdims=True)
    D = xx.T + yy - 2 * x.T @ y # broadcasting
    D[D<0.0] = 0.0 # due to numerics it can be smaller than zero
    return np.sqrt(D)

### $\varepsilon$ graph
- neighbors of $x_i$ = data points closer than $\varepsilon$ to $x_i$
- $\varepsilon$-graph is sensitive to the metric of the data points
- Sensitive to scaling
- Each point connects to all points within distance $\varepsilon$
- $\lVert x_i - x_j \rVert \leq \varepsilon$

$$
W_{ij} = 
\begin{cases} 
0 \hspace{0.4cm} \text{if } i=j \\
D_{ij} \hspace{0.2cm} \text{if } x_j \text{ and } x_j \text{ are neighbors} \\
\infty \hspace{0.3cm} \text{otherwise}  
\end{cases}
$$

In [119]:
def eps_graph(D, eps):
    # input: D are the non-squared distances
    n = D.shape[0]
    W = np.inf * np.ones((n, n)) # set all entries to infinity
    W.flat[::n+1] = 0.0 # set the diagonal to zero
    for i in range(n):
        eps_nb = (D[i] < eps)
        W[i, eps_nb] = D[i, eps_nb]  # set neighbors to their distance
    return W

### $k$ nearest neighbor graph (knn-graph)
- neighbors of $x_i$ = $k$-nearest data points of $x_i$
- $k$-nn does not depend on scaling
- Each point connects to its k closest points

$$
W_{ij} = 
\begin{cases} 
0 \hspace{0.4cm} \text{if } i=j \\
D_{ij} \hspace{0.2cm} \text{if } x_j \text{ and } x_j \text{ are neighbors} \\
\infty \hspace{0.3cm} \text{otherwise}  
\end{cases}
$$

In [120]:
def knn_graph(D, k, loop=0):
    # input: D are the non-squared distances
    n = D.shape[0]       # number of data points
    knn = D.argpartition(k, axis=1)[:,:k+1]
    W = np.inf * np.ones((n, n))   # set all entries to infinity
    if loop == 1:
        for i in range(n):
            W[i,knn[i]] = D[i,knn[i]]  # set neighbors to their distance
    elif loop == 0:
        all_rows = np.arange(n).reshape(n,1)
        W[all_rows, knn] = D[all_rows, knn]
    return W

In [121]:
n = 400
x, z = swissroll(n)
W = knn_graph(distances(x), 20)
#W = eps_graph(distances(x),3.0)
plot_graph_3d(x, W, z[0])

### Floyd-Warshall algorithm

Given a graph with edge weights $W_{ij}$ (their lengths) calculate a matrix $D$ of all lengths of all paths along the graph. Initialize $D=W$.

So compute $D_{ij} =$ shortest path between $i,j$

- update all lengths in $D$ with respect to paths via $x_1$
- update all lengths in $D$ with respect to paths via $x_2$
- $\ldots$
- update all lengths in $D$ with respect to paths via $x_n$


In [122]:
def floyd_warshall(W, loops=2):
    # assumes non-squared distances in W
    D = W.copy()
    n = D.shape[0]
    if loops==3:
        for k in range(n):
            for i in range(n):
                for j in range(n):
                    D[i,j] = min(D[i,j], D[i,k]+D[k,j])
    elif loops==2:
        # with two loops
        for k in range(n):
            for i in range(n):
                D[i,:] = np.minimum(D[i,:], D[i,k]+D[k,:])
    else:
        raise 'only implemented for at least two loops'
    return D

### Multi dimensional scaling

**Algortihm**

1. Calculate Gram matrix (inner product matrix) $\color{red}G=-\frac{1}{2}HDH$ and $H=I_n - \frac{1}{n} 1_n 1_n^\top$
2. Calculate EVD of $G=V\Lambda V^\top$
3. Pick $d$ largest eigenvalues from $\Lambda$ and collect corresponding eigenvectors as columns into the matrix $V_d$
4. Calculate data matrix $X=\Lambda^{\frac{1}{2}} V_d^\top$

In [123]:
def mds(D, d):
    # input:  D - distances (not squared)
    #         d - number of output dimensions
    # output: (d,n)-data matrix 
    n = D.shape[0]
    #H = np.eye(n) - np.ones((n, n))/n   # centering matrix
    #G = - H @ (D*D) @ H / 2             # Gram matrix
    DD = D*D
    DD = DD - DD.mean(axis=0, keepdims=True)   # center from the left
    DD = DD - DD.mean(axis=1, keepdims=True)   # center from the right
    G = - DD / 2
    Lambda, V = np.linalg.eigh(G)       # eigendecomposition
    # pick the last (largest) `d` eigenvalue and eigenvectors
    Lambda, V = Lambda[:-d-1:-1], V[:,:-d-1:-1]
    #x = np.diag(np.sqrt(Lambda)) @ V.T        # k dimensional embedding
    # let's use broadcasting, but first do the proper reshaping
    x = np.sqrt(Lambda).reshape(d,1) * V.T
    return x

In [124]:
# goal: create a map from a distance table scanned 
# if I recall correctly, I typed/ocr'ed them from the last page of an old road map
cities = ['Aachen', 'Basel', 'Berlin', 'Bremen', 'Dortmund', 
          'Dresden', 'Duesseldorf', 'Emden', 'Erfurt', 
          'Flensburg', 'Frankfurt M.', 'Frankfurt O.', 
          'Garmirsch-Patenkirchen', 'Goerlitz', 'Hamburg', 
          'Hannover', 'Kassel', 'Koblenz', 'Koeln', 'Leipzig', 
          'Mannheim', 'Muenchen', 'Nuernberg', 'Passau', 
          'Rostock', 'Saarbruecken', 'Salzburg', 'Stuttgart', 
          'Trier', 'Wiesbaden']
scanned_distances = \
'''545
650  875
370  775 400
155  555 495 235
645  745 200 490 515
 90  550 560 285  70 580
375  845 520 140 305 620 290
440  585 300 340 310 215 375 470 
625  980 450 275 490 660 540 400 520
255  335 550 445 225 460 225 520 260  650
700  940 105 460 560 180 625 590 370  550 610
700  370 675 835 700 550 680 930 510 1020 480 470
750  840 220 575 610 105 680 700 320  690 560 170 650
480  825 300 130 350 500 400 255 370  160 500 385 860 530
355  680 290 130 215 385 280 260 220  310 350 350 720 470 155
310  525 385 285 165 350 235 390 150  470 200 450 560 450 310 170
165  405 600 410 190 510 145 425 310  670 120 660 550 610 520 390 250
 75  505 580 315 100 570  40 330 370  570 200 640 650 670 430 300 250 105
570  710 190 390 440 115 505 520 140  570 385 255 520 215 440 290 280 440 500
290  270 615 515 300 530 280 600 330  725  85 680 410 630 570 430 270 150 250 460
650  390 590 750 605 460 610 840 420  930 400 650  90 560 780 630 470 490 580 430 350 
470  450 440 585 440 315 445 675 270  795 230 510 260 420 610 470 310 340 410 280 240 170
690  580 630 800 660 465 655 890 460  980 440 700 280 570 820 680 520 550 620 470 440 190 220
650 1000 230 300 515 440 570 425 490  285 740 325 870 470 180 330 560 690 600 380 810 780 630 820
310  265 725 550 330 640 280 560 440  810 200 790 500 740 660 530 380 180 250 570 140 430 370 570 920
800  530 735 910 755 605 770 980 580 1080 540 800 180 540 920 780 625 660 720 570 510 150 320 120 940 600
420  270 630 630 420 510 410 710 350  810 210 700 300 610 660 520 360 280 370 470 135 230 210 400 820 220 390
160  325 715 480 260 635 200 500 430  735 190 780 580 730 590 460 340 140 180 560 180 500 420 620 760 100 700 300
230  350 570 430 210 490 200 480 280  680  40 640 500 590 520 380 220  80 170 410 100 430 260 470 760 160 570 220 150'''


In [125]:
# convert them into a symmetric numpy matrix
n = len(cities)
D = np.zeros((n, n))
rows = scanned_distances.split('\n')
for i in range(n-1):
    D[i+1,:i+1] = rows[i].split()
D = D + D.T   # symmetrize
print(D[:5,:5])

x = mds(D, 2)
fig = px.scatter(x=x[0], y=x[1], text=cities, title='Germany')
fig.update_yaxes(scaleanchor = "x", scaleratio = 1)
fig.update_layout(width=800, height=800)
#fig.show()

# change plot 
fig = px.scatter(x=x[1], y=-x[0], text=cities, title='Germany')
fig.update_yaxes(scaleanchor = "x", scaleratio = 1)
fig.update_layout(width=800, height=800)
fig.show()

[[  0. 545. 650. 370. 155.]
 [545.   0. 875. 775. 555.]
 [650. 875.   0. 400. 495.]
 [370. 775. 400.   0. 235.]
 [155. 555. 495. 235.   0.]]


### ISOMAP stepwise

##### <u>Step 1: Neighborhood Graph</u>

$$
W_{ij} = 
\begin{cases} 
0 \hspace{0.4cm} \text{if } i=j \\
D_{ij}=\lVert x_i-x_j \rVert \hspace{0.2cm} \text{if } x_j \text{ and } x_j \text{ are neighbors according to } \begin{cases} k\text{-nn graph} \\ \varepsilon\text{-graph} \end{cases} \\
\infty \hspace{0.3cm} \text{otherwise}  
\end{cases}
$$

##### <u>Step 2: All-pairs shortest paths</u>

Given a graph with edge weights $W_{ij}$ (their lengths) calculate a matrix $D$ of all lengths of all paths along the graph. Initialize $D=W$.

So compute $D_{ij} =$ shortest path between $i,j$

Using: Floyd–Warshall

```python
def floyd_warshall(W):    # Python/numpy: 2 python loops
    # assumes non-squared distances in W
    D = W.copy()
    n = W.shape[0]
    for k in range(n):
        for i in range(n):
            D[i,:] = np.minimum(D[i,:], D[i,k]+D[k,:])
    return D

```

##### <u>Step 3: MDS (Multidimensional Scaling)</u>


**Algortihm**

1. Calculate Gram matrix (inner product matrix) $\color{red}G=-\frac{1}{2}HDH$
2. Calculate EVD of $G=V\Lambda V^\top$
3. Pick $d$ largest eigenvalues from $\Lambda$ and collect corresponding eigenvectors as columns into the matrix $V_d$
4. Calculate data matrix $X=\Lambda^{\frac{1}{2}} V_d^\top$


In [126]:
def isomap(x, d=2, k=None, eps=None):
    Euclidian_dist = distances(x)
    if k is not None:
        W = knn_graph(Euclidian_dist, k)
    elif eps is not None:
        W = eps_graph(Euclidian_dist, eps)
    else:
        raise "please set 'k' or 'eps'"
    Geodesic_dist = floyd_warshall(W)
    y = mds(Geodesic_dist, d)
    return y, W, Geodesic_dist

In [127]:
x = np.arange(10).reshape(1,10)   # 10 data points (scalars)
D = distances(x)**2
W = knn_graph(D, 3)
x = np.random.randn(3,4)
W = knn_graph(distances(x), 2)
n = 1000
x, z = swissroll(n, evenly_sampled=False)
k = 20
plot_graph_3d(x, knn_graph(distances(x), k), z[0]).show()
y, W, DD = isomap(x, d=3, k=k)
px.scatter(x=y[0], y=y[1], color=z[0])

<a class="anchor" id="lle"></a>
## 4. LLE aka Locally Linear Embedding

#### Core Idea
**Approximate manifold locally linearly $\rightarrow$ reconstruct low dimensional embedding matching local linear structure**

0. Given data matrix $X=[x_1,...,x_n] \in \mathbb{R}^{D \times n}$ with data points as columns
1. Build neighborhood graph
2. Find weights $W$ that locally reconstruct data linearly (i.e. express data points as local linear combination of their neighbors)
    - $x_i \approx \sum_{j} W_{ij} x_j \rightarrow \min_W \sum_i \lVert x_i - \sum_{j} W_{ij} x_j\rVert^2$ such that $\sum_{j} W_{ij} = 1$ for $i=1,...,n$
    - $W_{ij} = 0$ if $x_j$ is not a neighbor of $x_i$
3. Solve eigenvalue problem to get low dimensional embedding
    - Find low dimensional embedding $Z=[z_1,...,z_n]$ that minimizes $\sum_i \lVert z_i - \sum_{j} W_{ij} z_j\rVert^2$

So LLE = Graph $\rightarrow$ Local linear combinations $\rightarrow$ Eigenvalues

### LLE stepwise

<p align="center">
<img src="pics/lle.jpeg" width="500"/>
</p>



##### <u>Step 0: Data matrix</u>

Collect $n$ data points of shape $\mathbb{R}^D$ into columns of $X$

##### <u>Step 1: Neighborhood Graph</u>

**For each data $x_i$ select its neighbors, e.g. its $k$ nearest neighbors**

```python
for i=1:N
    compute the distance from Xi to every other point Xj
    find the K smallest distances 
    assign the corresponding points to be neighbours of Xi
end

```


##### <u>Step 2: Finding Weights</u>

Constraint optimization problem
$$
\min_W \sum_i \lVert x_i - \sum_j W_{ij}x_j \rVert^2 \\
\text{s.t. } \sum_j W_{ij}=1 \text{ for }i=1,...,n
$$
Can be solved for each data point $x$ with neighbors $\eta_j$ seperately
$$
\min_w \lVert x - \sum_j w_j\eta_j \rVert^2 \\
\text{s.t. } \sum_j w_j=1 
$$

Rewrite objekctive
$$
\frac{1}{2} \lVert x - \sum_j w_j \eta_j \rVert^2 = \frac{1}{2} \lVert \sum_j w_j (x-\eta_j) \rVert^2 \hspace{0.2cm} \text{ since } \sum_j w_j=1 = 1^\top w = 1 \\
= \frac{1}{2} \sum_{jk} w_j \underbrace{(x - \eta_j)^\top (x - \eta_k)}_{=: C_{jk}} w_k \\
= \frac{1}{2} w^\top C w \hspace{0.2cm} \text{ where matrix }C \text{ has enries }C_{jk}
$$

Lagrangian

$$
L(w, \lambda) = \frac{1}{2} w^\top C w - \lambda(1^\top w - 1) \\
$$
Taking derivative:
$$
dL = (w^\top C - \lambda 1^\top) dw
$$
Setting gradient to zero gives:
$$
w = \lambda C^{-1} 1
$$
Component-wise:

$$
w_j = \lambda \sum_k (C^{-1})_{jk}
$$
- choose normalizing constant $\lambda$ such that $1^\top w = 1$
    - solve $Cw=1$ then enforce $1^\top w = 1$

**Summary**
- Find weight vector for single data point $x_i$
    - 1. find neares neighbors $\eta_1,...,\eta_k$
    - 2. compute local covariance matrix $C$ with emntries $(x - \eta_j)^\top (x - \eta_k)$
    - 3. D´define weights $w$ according to $Cw=1$ and $1^\top w = 1$
- Combine weight vectors into matrix $W \in \mathbb{R}^{n \times n}$
    - 1. compute weight vectors for all data points
    - 2. for a data point $x_i$ store weights for each neighbor in the appropriate entries in the $i-th$ column of $W$ and fill zeros for non-neighbors

```python

for i=1:N
    create matrix Z consisting of all neighbours of Xi
    subtract Xi from every column of Z
    compute the local covariance C=Z'*Z
    solve linear system C*w = 1 for w
    set Wij=0 if j is not a neighbor of i
    set the remaining elements in the ith row of W equal to w/sum(w);
end

```

##### <u>Step 2: Finding embedding</u>

Constraint optimization problem
$$
\min_Y \sum_i \lVert y_i - \sum_j W_{ij}y_j \rVert^2 \\
\text{s.t. } Y1=0 \text{ zero mean } \\
YY^\top = I \text{ covariance identity }
$$
Can be rewritten as
$$
\min_Y tr YMY^\top \\
\text{s.t. } Y1=0  \\
YY^\top = I 
M = I - W^\top - W + W^\top W = (I-W)^\top(I-W)
$$
$\Rightarrow$ Solving this yields that rows of $Y$ are eigenvectors to the smallest eigenvalues 

```python
create sparse matrix M = (I-W)'*(I-W)
    find bottom d+1 eigenvectors of M 
        #(corresponding to the d+1 smallest eigenvalues) 
    set the qth ROW of Y to be the q+1 smallest eigenvector 
        #(discard the bottom eigenvector [1,1,1,1...] with eigenvalue zero)
```

In [128]:
def lle(x, d=2, k=9):
    # 1. find nearest neighbors
    d, n = x.shape[0], x.shape[1]
    dist = distances(x)          # non-squared distances
    dist.flat[::n+1] = np.inf  # you are not your own neighbor!
    knn = dist.argpartition(k-1, axis=1)[:,:k]
    W = np.zeros((n,n))
    for i in range(n):
        W[i,knn[i]] = dist[i,knn[i]]
    plot_graph(x, W).show()
    # 2. solve for reconstruction weights
    if k>d:
        print('note: since k>d we regularize')
        tol = 1.0e-3
    else:
        tol = 0.0
    W = np.zeros((n, n))
    for i in range(n):
        z = x[:,knn[i]] - x[:,i].reshape(d, 1)
        C = z.T @ z
        C = C + np.eye(k)*tol*np.trace(C)
        w = np.linalg.solve(C, np.ones(k))
        W[i,knn[i]] = w / w.sum()
    # 3. compute embedding coordinates
    M = (np.eye(n) - W).T @ (np.eye(n) - W)
    Lambda, V = np.linalg.eigh(M)
    y = V[:,1:3].T * np.sqrt(n)
    return y, W

In [129]:
n = 200
x, z = swissroll(200, evenly_sampled=False)

y, W = lle(x, d=2, k=9)
plot_graph_3d(x, W).show()   # check the graph
px.scatter(x=y[0], y=y[1], color=z[0])

note: since k>d we regularize


<a class="anchor" id="lib1"></a>
# 5. ISOMAP library

```python
# 1. Import Isomap
from sklearn.manifold import Isomap

isomap1 = Isomap(
    n_components=2,
    n_neighbors=5, # number of neighbors (k-NN graph)
    metric="euclidean",
    path_method="auto" # "auto", "FW", "D"
)

X_iso = isomap1.fit_transform(X)


# 2. Basic Usage
isomap2 = Isomap(n_components=2, n_neighbors=10)
X_iso = isomap2.fit_transform(X)


# 3. Access Geodesic Distance Matrix
isomap3 = Isomap(n_neighbors=5)
X_iso = isomap3.fit_transform(X)

D_geo = isomap3.dist_matrix_ # geodesic distances (shortest paths)


# 4. Change Distance Metric
isomap4 = Isomap(
    n_components=2,
    n_neighbors=5,
    metric="manhattan" # or "cosine", "minkowski"
)

X_iso = isomap4.fit_transform(X)


# 5. Pipeline (recommended)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("isomap", Isomap(n_components=2, n_neighbors=5))
])

X_pipeline = pipeline.fit_transform(X)


# 6. Isomap + Classification
from sklearn.svm import SVC

pipeline2 = Pipeline([
    ("scaler", StandardScaler()),
    ("isomap", Isomap(n_components=2, n_neighbors=5)),
    ("clf", SVC())
])

pipeline2.fit(X, y)
pipeline2.score(X, y)


# 7. Visualization (2D embedding)
import matplotlib.pyplot as plt

isomap7 = Isomap(n_components=2, n_neighbors=5)
X_2d = isomap7.fit_transform(X)

plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("Isomap Projection")
plt.show()


# 8. Tune number of neighbors (k)
results = []

for k in [3, 5, 10, 20]:
    iso = Isomap(n_components=2, n_neighbors=k)
    X_k = iso.fit_transform(X)
    results.append(X_k.shape)

print(results)


# 9. Reconstruction Error (quality measure)
isomap9 = Isomap(n_neighbors=5)
X_iso = isomap9.fit_transform(X)

error = isomap9.reconstruction_error()
print("Reconstruction error:", error)


# 10. Isomap on Nonlinear Data
from sklearn.datasets import make_swiss_roll

X, color = make_swiss_roll(n_samples=500)

iso = Isomap(n_components=2, n_neighbors=10)
X_iso = iso.fit_transform(X)
```

In [ ]:
# Generate S-curve data
X, color = make_s_curve(n_samples=1000, noise=0.05, random_state=42)

df_3d = pd.DataFrame(X, columns=["x", "y", "z"])
df_3d["color"] = color

# 3D visualization
fig = px.scatter_3d(df_3d,x="x", y="y", z="z",color="color",title="S-Curve Dataset (3D)")
fig.show()


# ISOMAP
isomap = Isomap(n_neighbors=10, n_components=2)
X_iso = isomap.fit_transform(X)

df_iso = pd.DataFrame(X_iso, columns=["dim1", "dim2"])
df_iso["color"] = color

fig_iso = px.scatter(df_iso,x="dim1", y="dim2",color="color",title="Isomap Projection (2D)")
fig_iso.show()



<a class="anchor" id="lib2"></a>
# 6. LLE library

```python
# 1. Import LLE
from sklearn.manifold import LocallyLinearEmbedding

lle1 = LocallyLinearEmbedding(
    n_components=2,
    n_neighbors=10,
    method="standard" # "standard", "modified", "hessian", "ltsa"
)

X_lle = lle1.fit_transform(X)


# 2. Basic Usage
lle2 = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_lle = lle2.fit_transform(X)


# 3. Different LLE Variants
lle_standard = LocallyLinearEmbedding(method="standard")
lle_modified = LocallyLinearEmbedding(method="modified")
lle_hessian = LocallyLinearEmbedding(method="hessian")
lle_ltsa = LocallyLinearEmbedding(method="ltsa")

X_std = lle_standard.fit_transform(X)
X_mod = lle_modified.fit_transform(X)


# 4. Access Reconstruction Error
lle4 = LocallyLinearEmbedding(n_neighbors=10)
X_lle = lle4.fit_transform(X)

error = lle4.reconstruction_error_
print("Reconstruction error:", error)


# 5. Pipeline (recommended)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lle", LocallyLinearEmbedding(n_components=2, n_neighbors=10))
])

X_pipeline = pipeline.fit_transform(X)


# 6. LLE + Classification
from sklearn.svm import SVC

pipeline2 = Pipeline([
    ("scaler", StandardScaler()),
    ("lle", LocallyLinearEmbedding(n_components=2, n_neighbors=10)),
    ("clf", SVC())
])

pipeline2.fit(X, y)
pipeline2.score(X, y)


# 7. Visualization (2D embedding)
import matplotlib.pyplot as plt

lle7 = LocallyLinearEmbedding(n_components=2, n_neighbors=10)
X_2d = lle7.fit_transform(X)

plt.scatter(X_2d[:, 0], X_2d[:, 1], c=y)
plt.xlabel("Component 1")
plt.ylabel("Component 2")
plt.title("LLE Projection")
plt.show()


# 8. Tune number of neighbors
results = []

for k in [5, 10, 15, 20]:
    lle = LocallyLinearEmbedding(n_components=2, n_neighbors=k)
    X_k = lle.fit_transform(X)
    results.append(X_k.shape)

print(results)


# 9. LLE on Nonlinear Data
from sklearn.datasets import make_swiss_roll

X, color = make_swiss_roll(n_samples=500)

lle9 = LocallyLinearEmbedding(n_components=2, n_neighbors=12)
X_lle = lle9.fit_transform(X)


# 10. Compare LLE variants
methods = ["standard", "modified", "hessian", "ltsa"]

for m in methods:
    lle = LocallyLinearEmbedding(n_components=2, n_neighbors=10, method=m)
    X_m = lle.fit_transform(X)
    print(m, X_m.shape)
```

In [134]:
# 3D visualization
fig = px.scatter_3d(df_3d,x="x", y="y", z="z",color="color",title="S-Curve Dataset (3D)")
fig.show()

lle = LocallyLinearEmbedding(n_neighbors=10, n_components=2, method="standard")
X_lle = lle.fit_transform(X)

df_lle = pd.DataFrame(X_lle, columns=["dim1", "dim2"])
df_lle["color"] = color

fig_lle = px.scatter(df_lle,x="dim1", y="dim2",color="color",title="LLE Projection (2D)")
fig_lle.show()